Experiment date and environment

In [1]:
date_n_env = "optuna_apr6_env1" # Experiment date and environment

In [2]:
# Import libraries
import pygame
import gymnasium as gym
import numpy as np
import copy
import itertools
np.random.seed(33) # seeding

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin,2)
    return dec

# Function to check if a point is inside a polygon (Ray-casting algorithm)
def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to return minimum distance in a list of points
def min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1-p2)
        dists.append(dist)
    return float(np.min(dists))

In [4]:
experiments_path = r'./experiment_sets.txt'
# Read the experiments file and select the experiment
with open(experiments_path, 'r') as experiment_file:
    codes = experiment_file.read()
    exec(codes) # execute
selected_experiment = set1 # Select the experiment set
selected_experiment

{'field': [(43.0, 24.0), (20.0, 47.0), (13.0, 34.0), (12.0, 9.0), (22.0, 0.0)],
 'init_positions': [array([14., 34.]), array([40., 25.]), array([20., 40.])],
 'infected_locations': {(15.0, 10.0),
  (22.0, 13.0),
  (25.0, 20.0),
  (25.0, 35.0),
  (26.0, 21.0),
  (35.0, 25.0)}}

In [5]:
sf = 10
selected_experiment['field'] = [(x*sf, y*sf) for (x,y) in selected_experiment['field']]
selected_experiment['infected_locations'] = [(x*sf, y*sf) for (x,y) in selected_experiment['infected_locations']]
selected_experiment['init_positions'] = [v*sf for v in selected_experiment['init_positions']]
selected_experiment, len(selected_experiment['init_positions']), len(np.unique(selected_experiment['init_positions'], axis=0))

({'field': [(430.0, 240.0),
   (200.0, 470.0),
   (130.0, 340.0),
   (120.0, 90.0),
   (220.0, 0.0)],
  'init_positions': [array([140., 340.]),
   array([400., 250.]),
   array([200., 400.])],
  'infected_locations': [(260.0, 210.0),
   (150.0, 100.0),
   (220.0, 130.0),
   (250.0, 350.0),
   (350.0, 250.0),
   (250.0, 200.0)]},
 3,
 3)

In [6]:
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}
    def __init__(self, render_mode=None, poly_vertices=copy.deepcopy(selected_experiment['field'])):
        super(MultiRobotEnv, self).__init__()

        # Screen dimensions
        self.edge_buffer = 10 # Boundary above the max values
        self.xs, self.ys = zip(*poly_vertices) # x and y values of the vertices of the polygonal field
        self.WIDTH, self.HEIGHT = 1000, 1000 # Use this if we want to have fixed width and height. Default: 800x600        
        # self.WIDTH, self.HEIGHT = max(self.xs) + self.edge_buffer, max(self.ys) + self.edge_buffer
        self.poly_vertices = poly_vertices # Vertices of polygon

        # Number of robots
        self.num_robots = 3 # Rendering error if more than 7

        # Robot parameters
        self.robot_size = 10
        self.mass = 1.0
        # self.g = 0.1  # Gravity or directional force
        self.thrust_power = 0.5  # Force applied per action
        self.max_speed = 5  # Maximum speed    
        self.min_speed = -5 # Minimum speed
        self.min_positions = np.zeros(self.num_robots*2) # Minimum positions
        self.max_positions = np.array([[self.WIDTH, self.HEIGHT] for _ in range(self.num_robots)]) # Maximum positions
        self.min_velocities = np.array([[self.min_speed, self.min_speed] for _ in range(self.num_robots)]) # Min speed list
        self.max_velocities = np.array([[self.max_speed, self.max_speed] for _ in range(self.num_robots)]) # Max speed list

        # infected locations
        self.infected_size = 10 # Radius of infected locations
        self.infected_length = len(copy.deepcopy(selected_experiment['infected_locations']))
        self.infected_state_length = 2**(self.infected_length) # 2**5, binary to decimal

        # Action space: thrust in x and y directions for each robot
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.num_robots, 2), dtype=np.float32)

        # Observation space: position and velocity (x, y, vx, vy) for each robot + infected location        
        self.observation_space = gym.spaces.Box( # The (visited) weed locations are tracked on the observation space
                    low = np.concatenate((self.min_positions.flatten(), self.min_velocities.flatten(), np.array([0]))), # Lowest positions and velocities
                    high = np.concatenate((self.max_positions.flatten(), self.max_velocities.flatten(), np.array([self.infected_state_length - 1]))), # highest positions and velocities
                    dtype=np.float32)

        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check if the render mode is correct
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        # If human-rendering is used, `self.screen` will be a reference to the screen that we draw to. `self.clock` will be a clock that is used
        # to ensure that the environment is rendered at the correct framerate in human-mode. They will remain `None` until human-mode is used for the first time.   

        # Reset the environment and start
        self.reset()
    
    def _get_obs(self):
        info = {f'robot{i}': self.robot_positions[i] for i in range(self.num_robots)} # Current position of each robot
        infected = binary_list_to_decimal(list(self.infected_dict.values())) # Convert the binary list of infected locations to a decimal value
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(), np.array([infected])), dtype=np.float32) # Current state of the robots
        return state, info        

    def reset(self, seed=None, options={}):
        # Reset the visited states and counts
        self.step_count = 0
        self.visited = set()
        self.infected_locations = copy.deepcopy(selected_experiment['infected_locations'])
        self.infected_dict = {v:0 for v in self.infected_locations} # 0 for unvisited infected locations, 1 for visited
        self.robot_positions = np.array(copy.deepcopy(selected_experiment['init_positions']))[:self.num_robots] # Initial positions of each robot
        self.robot_velocities = np.zeros((self.num_robots, 2)) # Initial velocities of each robot (zero)
        return self._get_obs()
    
    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1
        for i in range(self.num_robots): # For every robot
            ax, ay = actions[i] * self.thrust_power # What actions to take

            # Update velocity
            self.robot_velocities[i][0] += ax / self.mass
            self.robot_velocities[i][1] += ay / self.mass # + self.g if we want to have directional force along y axis

            # Limit velocity
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Predict new position
            new_position = self.robot_positions[i] + self.robot_velocities[i]

            # Boundary conditions (keep robot within polygon)
            if is_inside_polygon(new_position, self.poly_vertices):
                pass
            else: # Hits the wall!
                rewards -= 10 # Medium negative reward for hitting the wall
                self.robot_velocities[i][:] = 0 # Stop movement

            # Update position
            self.robot_positions[i] += self.robot_velocities[i]
            
            # Boundary conditions (keep robot within screen)
            self.robot_positions[i] = np.clip(self.robot_positions[i], [0, 0], [self.WIDTH, self.HEIGHT])

            # Check if location is visited before, and add it to the visited locations
            if tuple(self.robot_positions[i]) in self.visited:
                rewards -= 10 # Small negative reward for visiting previous location
            else:
                rewards -= 1 # Very small negative reward for visiting new locations
            self.visited.add(tuple(self.robot_positions[i]))            

            # Check if any infected location is visited        
            nearby_infected_locations = [] # To store the nearby infected locations
            for j, inf_loc in enumerate(self.infected_locations): # Loop through each infected location
                dist = np.linalg.norm(self.robot_positions[i]-inf_loc) # Distance between robot position and infected location
                if dist <= self.infected_size: # If the distance is within the radius of the infected location size
                    nearby_infected_locations.append(inf_loc) # Add the infected location
                    rewards += 100 # Medium positive rewards for visiting each infected location
                    # input("Pause!") # Only pause if you want to visualize visiting infected locations
            for inf_loc in nearby_infected_locations:
                self.infected_locations.remove(inf_loc) # Delete each visited infected location
                self.infected_dict[tuple(inf_loc)] = 1 # Update the infected dictionary
        
        # Check if all infected locations are visited
        if len(self.infected_locations) == 0:
            rewards += 100000 # Big positive rewards for visiting all infected locations
            terminated = True
        
        # Check if any collisions occurred
        if self.num_robots > 1:
            min_dist_between_robots = min_dist(self.robot_positions) # Minimum distance between robots
            if min_dist_between_robots < self.robot_size:
                rewards -= 100000 # Big negative rewards for collisions
                terminated = True

        obs, info = self._get_obs() # Get the updated observations
        # rewards = rewards * self.gamma ** self.step_count
        return obs, rewards, terminated, truncated, info
    
    def render(self):
        # Initialize pygame
        if self.screen is None and self.render_mode == "human": # Initialize pygame if it is not initialized
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True
        
        self.screen.fill((255, 255, 255)) # White color for the background
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128)]  # Colors for each robot: Red, Green, Blue, Orange, Violet, Pink, Grey
        pix_size = 10

        # Draw the polygon
        # pixel_poly_vertices = [(point[0] * pix_size, point[1] * pix_size) for point in self.poly_vertices]
        pygame.draw.polygon(surface=self.screen, 
                            color=(255, 255, 0), # Yello color for the polygon
                            points=self.poly_vertices)
        
        # Draw the visited regions
        for point in self.visited:
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), point, pix_size/2) # Light grey color for visited regions, with transparency alpha

        # Draw robots
        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, colors[i], (int(self.robot_positions[i][0]), int(self.robot_positions[i][1])), pix_size/2) # Pick the colors from above list

        # Draw infected locations
            for l in self.infected_locations:
                pygame.draw.circle(self.screen, (0, 255, 255), (int(l[0]), int(l[1])), pix_size/2) # Cyan color for infected locations
        
        pygame.display.flip() # Allows only a portion of the screen to be updated
        self.clock.tick(60)
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [7]:
# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=1000)

In [8]:
assert False, "Manual pause"

AssertionError: Manual pause

**Training**

In [9]:
import stable_baselines3
print(stable_baselines3.__version__)

2.6.0a1


In [10]:
import sb3_contrib
print(sb3_contrib.__version__)

2.6.0a1


In [17]:
# Import libraries
import optuna
import matplotlib.pyplot as plt
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env # for making environments
from stable_baselines3.common.callbacks import LogEveryNTimesteps, EvalCallback, StopTrainingOnNoModelImprovement
import traceback

In [18]:
# Experimental settings
weights_path = rf"./new_models/{date_n_env}"
tensorboard_log_path = rf"./tensorboard_logs/optuna/{date_n_env}"
time_steps = 1e6
n_trials = 20 # Number of trials

# Callbacks
logger = LogEveryNTimesteps(n_steps=10000)
stop_train_callback = StopTrainingOnNoModelImprovement(max_no_improvement_evals=100, min_evals=5, verbose=1)

In [ ]:
%%time
from sb3_contrib import TRPO

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)

    # Train the TRPO model with sampled hyperparameters
    model = TRPO("MlpPolicy",
                vec_env,
                n_steps=n_steps,
                gamma=gamma,
                learning_rate=learning_rate,
                gae_lambda=gae_lambda,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_trpo")

    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
        vec_env.reset()
        eval_env.reset()
        # Evaluate model performance
        mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
        mean_rewards.append(mean_reward)
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
        vec_env.reset()
        eval_env.reset()
        mean_reward = -1e5
        mean_rewards.append(mean_reward)
    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [ ]:
%%time
from stable_baselines3 import PPO

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)

    # Train the PPO model with sampled hyperparameters
    model = PPO("MlpPolicy",
                vec_env,
                n_steps=n_steps,
                gamma=gamma,
                learning_rate=learning_rate,
                gae_lambda=gae_lambda,
                ent_coef=ent_coef,
                vf_coef=vf_coef,
                max_grad_norm = max_grad_norm,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_ppo")

    # model.learn(total_timesteps=time_steps, callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
    vec_env.reset()
    eval_env.reset()

    # Evaluate model performance
    mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
    mean_rewards.append(mean_reward)

    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [ ]:
%%time
from stable_baselines3 import A2C

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    n_steps = trial.suggest_categorical("n_steps", [5, 10, 20])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)

    # Train the A2C model with sampled hyperparameters
    model = A2C("MlpPolicy",
                vec_env,
                n_steps=n_steps,
                gamma=gamma,
                learning_rate=learning_rate,
                gae_lambda=gae_lambda,
                ent_coef=ent_coef,
                vf_coef=vf_coef,
                max_grad_norm = max_grad_norm,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_a2c")

    # model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
    vec_env.reset()
    eval_env.reset()

    # Evaluate model performance
    mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
    mean_rewards.append(mean_reward)

    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [ ]:
%%time
from sb3_contrib import ARS

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    # n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    # gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    # ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    # vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    # gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    # max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)

    # Train the ARS model with sampled hyperparameters
    model = ARS("LinearPolicy",
                vec_env,
                learning_rate=learning_rate,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_ars")

    # model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
    vec_env.reset()
    eval_env.reset()

    # Evaluate model performance
    mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
    mean_rewards.append(mean_reward)

    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [ ]:
%%time
from sb3_contrib import CrossQ

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    # n_steps = trial.suggest_categorical("n_steps", [5, 10, 20])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    # ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    # vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    # gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    # max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)
    buffer_size = trial.suggest_int('buffer_size', 1000, 100000, step=1000)

    # Train the CrossQ model with sampled hyperparameters
    model = CrossQ("MlpPolicy",
                vec_env,
                gamma=gamma,
                learning_rate=learning_rate,
                buffer_size = buffer_size,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_CrossQ")

    # model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
    vec_env.reset()
    eval_env.reset()

    # Evaluate model performance
    mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
    mean_rewards.append(mean_reward)

    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [ ]:
%%time
from sb3_contrib import TQC

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=4)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    # n_steps = trial.suggest_categorical("n_steps", [5, 10, 20])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    # ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    # vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    # gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    # max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)
    buffer_size = trial.suggest_int('buffer_size', 1000, 100000, step=1000)

    # Train the TQC model with sampled hyperparameters
    model = TQC("MlpPolicy",
                vec_env,
                gamma=gamma,
                learning_rate=learning_rate,
                buffer_size = buffer_size,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_TQC")

    # model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
    vec_env.reset()
    eval_env.reset()

    # Evaluate model performance
    mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
    mean_rewards.append(mean_reward)

    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

In [22]:
%%time
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

# Create the environment
vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=1)
eval_env = gym.make('MultiRobotEnv-v0') # Separate evaluation env
eval_callback = EvalCallback(eval_env, eval_freq=50000, callback_after_eval=stop_train_callback, verbose=1)

mean_rewards = []
best_reward = -1e10

# The noise objects for TD3
n_actions = vec_env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Objective function for optimization
def objective(trial):
    global best_reward
    # Suggest hyperparameters
    # n_steps = trial.suggest_categorical("n_steps", [5, 10, 20])
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True)
    # ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    # vf_coef = trial.suggest_float("vf_coef", 0.2, 0.7)
    # gae_lambda = trial.suggest_float("gae_lambda", 0.9, 1.0)
    # max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 0.99)
    buffer_size = trial.suggest_int('buffer_size', 1000, 100000, step=1000)

    # Train the TD3 model with sampled hyperparameters
    model = TD3("MlpPolicy",
                vec_env,
                action_noise=action_noise,
                # gamma=gamma,
                # learning_rate=learning_rate,
                # buffer_size = buffer_size,
                verbose=0,
                tensorboard_log=tensorboard_log_path+"_TD3")

    # model.learn(total_timesteps=time_steps,  callback=[eval_callback, logger])
    try:
        model.learn(total_timesteps=time_steps, callback=[logger])
        vec_env.reset()
        eval_env.reset()
        # Evaluate model performance
        mean_reward, _ = evaluate_policy(model, vec_env, n_eval_episodes=10, deterministic=True)
        mean_rewards.append(mean_reward)
    except Exception as e:
        print("Training failed with error:")
        traceback.print_exc()  # Optional: prints the full stack trace
        print("*"*50)
        vec_env.reset()
        eval_env.reset()
        mean_reward = -1e5 if not mean_rewards else min(mean_rewards)
        mean_rewards.append(mean_reward)
    if best_reward < mean_reward:
        best_reward = mean_reward
        # model.save(weights_path)
    return mean_reward

# Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=n_trials)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)
print("Best mean reward:", best_reward)

scaled_rewards = [x/100000 for x in mean_rewards]
x_vals = list(range(len(mean_rewards)))
plt.figure()
plt.xticks(x_vals)
plt.plot(scaled_rewards)
plt.ylabel("x$10^6$")
plt.show()
mean_rewards

[I 2025-04-09 11:14:30,453] A new study created in memory with name: no-name-2550a78c-1c7a-4bce-8f03-7ac77567bf7e
[W 2025-04-09 11:15:22,474] Trial 0 failed with parameters: {'gamma': 0.9076684542521628, 'learning_rate': 0.008928694346394503, 'buffer_size': 49000} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<timed exec>", line 41, in objective
  File "c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\stable_baselines3\td3\td3.py", line 222, in learn
    return super().learn(
           ^^^^^^^^^^^^^^
  File "c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\stable_baselines3\common\off_policy_algorithm.py", line 347, in learn
    self.train(batch_size=self.batch_size, gradient_steps=gradient_steps)
  File "c:\Users\choton\miniconda3\envs\r

KeyboardInterrupt: 

In [15]:
%%time
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

vec_env = make_vec_env('MultiRobotEnv-v0', n_envs=1)

# The noise objects for TD3
n_actions = vec_env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = TD3("MlpPolicy", 
              vec_env,
              action_noise=action_noise, 
              verbose=1, 
              tensorboard_log=tensorboard_log_path+"_TD3")
model.learn(total_timesteps=1000, callback=logger)
model.save(weights_path+"_TD3")
del model, TD3, make_vec_env

Using cuda device
Logging to ./tensorboard_logs/optuna/optuna_apr6_env1_TD3\TD3_21


c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


CPU times: total: 12.5 s
Wall time: 9.5 s


c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\stable_baselines3\common\save_util.py:284: UserWarning: Path 'new_models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")
